# Ana 10

In [1]:
"""
case3_main_figure_full.py
======================================================================
Case3 SHARE-seq 跨器官 GRN 分析 — 完整版 (主图 + 各子图单独保存)

主图 8 panel (4x2, 仅 skin vs brain; lung 在补充图):
  A UMAP(伪时序)  B Jaccard矩阵   C 边分解   D peak特异性
  G 分化度分布     H Nfib subnetwork  J GO双向对比  K 靶基因Euler

设计: 每个 panel 的绘制逻辑抽成独立函数 draw_X(ax, ...), 主图与单图
      调用同一函数 -> 两者完全一致, 无重复代码。

输出:
  Case3_main_figure.png/.pdf          (8 panel 主图)
  panels/panel_A.png/.pdf ... panel_K  (各子图单独)
  tables/...                           (导出表)

继承全部约束: 权重饱和不用(只用边/count/坐标)、无偏验证、lung降级、
             命名Title-case统一、活跃peak用value分位、GO读真实CSV。
依赖: numpy pandas scipy matplotlib seaborn; 可选 anndata
======================================================================
"""

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ---- 全局字体: 统一 16 号 ----
plt.rcParams.update({
    "font.size": 16,
    "axes.titlesize": 16,
    "axes.labelsize": 16,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "legend.fontsize": 16,
    "figure.titlesize": 16,
})
import matplotlib.gridspec as gridspec
from matplotlib.patches import Circle
import seaborn as sns
from collections import Counter, defaultdict
from scipy.stats import hypergeom

try:
    import anndata as ad
    _HAS_ANNDATA = True
except ImportError:
    _HAS_ANNDATA = False

# ====================================================================
# 配置
# ====================================================================
data_root = "/home/wuyan/dygmamba_project/NewRealPlan/case3/data/process/"

data_out = "/home/wuyan/dygmamba_project/DRIMA/data/case3/"
output_dir = data_out

TISSUES = {"skin": {"path": data_root + "skin/process/", "color": "#E41A1C"},
           "brain": {"path": data_root + "brain/process/", "color": "#377EB8"},
           "lung": {"path": data_root + "lung/process/", "color": "#4DAF4A"}}
ALL_TISSUES = ["skin", "brain"]      # 主图仅 skin/brain; lung 归补充

os.makedirs(output_dir, exist_ok=True)
os.makedirs(output_dir + "panels/", exist_ok=True)
os.makedirs(output_dir + "tables/", exist_ok=True)

SKIN_TFS = ["Dlx3", "Sox9", "Lef1", "Nfib", "Foxq1",
            "Hoxc13", "Msx2", "Gata3", "Tcf7", "Vdr"]
NON_TF_BLACKLIST = {"Ctcf", "Yy1", "Rad21", "Smc1a", "Smc3"}
REP_TF = "Nfib"
ACTIVE_VALUE_QUANTILE = 0.5
PEAK_OVERLAP_FRAC = 0.5
SKIN_C, BRAIN_C = TISSUES["skin"]["color"], TISSUES["brain"]["color"]


# ====================================================================
# 工具
# ====================================================================
def normalize_mouse_symbol(s):
    if not isinstance(s, str) or s == "" or s.startswith("chr"):
        return s
    if not s.isupper():
        return s
    return "-".join(p[:1].upper() + p[1:].lower() if p else p for p in s.split("-"))


def load_grn(path, load_dyn=False):
    res = {}
    for fn, k in [("pred_tf_gene.pkl", "avg"), ("pred_peak_gene.pkl", "rg")]:
        fp = os.path.join(path, fn)
        if os.path.exists(fp):
            res[k] = pd.read_pickle(fp)
    if load_dyn:
        fp = os.path.join(path, "pred_time_tf_gene.pkl")
        if os.path.exists(fp):
            res["dyn"] = pd.read_pickle(fp)
    if _HAS_ANNDATA:
        fp = os.path.join(path, "rna_processed.h5ad")
        if os.path.exists(fp):
            try:
                res["adata"] = ad.read_h5ad(fp)
            except Exception:
                pass
    return res


def clean_pairs(avg):
    df = avg.copy()
    df = df[~df["Gene"].astype(str).str.startswith("chr")]
    df = df[~df["TF"].astype(str).str.startswith("chr")]
    return df


def get_pairs(avg):
    df = clean_pairs(avg)
    return set(zip(df["TF"], df["Gene"]))


def get_target_sets(avg):
    df = clean_pairs(avg)
    df = df[~df["TF"].isin(NON_TF_BLACKLIST)]
    return df.groupby("TF")["Gene"].apply(set).to_dict()


def get_topN_pairs(avg, n):
    df = clean_pairs(avg)
    if "avg_ts_weight" in df.columns:
        df = df.sort_values("avg_ts_weight", ascending=False).head(n)
    else:
        df = df.head(n)
    return set(zip(df["TF"], df["Gene"]))


def parse_peak(p):
    try:
        s = str(p).replace(":", "-").split("-")
        return (s[0], int(s[1]), int(s[2]))
    except Exception:
        return None


def get_active_ivs(rg):
    if "value" in rg.columns:
        thr = rg["value"].quantile(ACTIVE_VALUE_QUANTILE)
        active = rg[rg["value"] > thr]
    else:
        active = rg
    peaks = active["Peak"][active["Peak"].astype(str).str.startswith("chr")].unique()
    ivs = [parse_peak(p) for p in peaks]
    return sorted([iv for iv in ivs if iv], key=lambda x: (x[0], x[1]))


def peak_spec_coord(tissue_ivs):
    names = [t for t in tissue_ivs if tissue_ivs[t]]
    if not names:
        return {}
    all_iv = sorted([(c, s, e) for t in names for c, s, e in tissue_ivs[t]])
    union = []
    for c, s, e in all_iv:
        if union and union[-1][0] == c and s <= union[-1][2]:
            union[-1] = (c, union[-1][1], max(union[-1][2], e))
        else:
            union.append((c, s, e))
    per = defaultdict(list)
    for t in names:
        for c, s, e in tissue_ivs[t]:
            per[(t, c)].append((s, e))
    spec = Counter()
    for uc, us, ue in union:
        n = sum(1 for t in names
                if any(min(ue, e) - max(us, s) > 0 for s, e in per.get((t, uc), [])))
        spec[n] += 1
    return dict(spec)


def identity_divergence(skin_sets, brain_sets):
    rows = []
    for tf in (set(skin_sets) & set(brain_sets)) - NON_TF_BLACKLIST:
        a, b = skin_sets[tf], brain_sets[tf]
        if not a and not b:
            continue
        inter, union = len(a & b), len(a | b)
        jac = inter / union if union else 0
        rows.append({"TF": tf, "n_skin": len(a), "n_brain": len(b),
                     "n_shared": inter, "divergence": 1 - jac})
    return pd.DataFrame(rows).set_index("TF").sort_values("divergence", ascending=False)


def compute_count_activity_curves(dyn, tf_list, n_bins=20):
    """
    沿伪时序计算选定 TF 的靶标数(count)活性曲线。
    [重要] 不用权重 (avg_weight 饱和), 只数每个 ts-bin 内该 TF 的去重靶标数。
    只对 tf_list 中的 TF 计算 (内存友好, 不遍历全部 847 个)。
    返回: dict {TF: np.array(n_bins,)} min-max 归一化后的曲线。
    """
    if dyn is None or "ts" not in dyn.columns:
        return {}
    tf_low = {t.lower(): t for t in tf_list}
    sub = dyn[dyn["TF"].astype(str).str.lower().isin(tf_low)].copy()
    if len(sub) == 0:
        return {}
    sub = sub[~sub["Gene"].astype(str).str.startswith("chr")]
    ts_min, ts_max = sub["ts"].min(), sub["ts"].max()
    sub["bin"] = pd.cut(sub["ts"], bins=np.linspace(ts_min, ts_max, n_bins + 1),
                        labels=False, include_lowest=True)
    curves = {}
    for tf_l, grp in sub.groupby(sub["TF"].astype(str).str.lower()):
        per_bin = grp.groupby("bin")["Gene"].nunique()
        curve = np.array([per_bin.get(i, 0) for i in range(n_bins)], dtype=float)
        # 轻度预平滑 (滑动平均窗口3) 去除 bin 抖动, 再归一化
        if len(curve) >= 3:
            kern = np.ones(3) / 3
            curve = np.convolve(curve, kern, mode="same")
        rng = curve.max() - curve.min()
        curve_norm = (curve - curve.min()) / rng if rng > 1e-9 else curve * 0
        curves[tf_low[tf_l]] = curve_norm
    return curves


def run_go_enrichment(gene_list, background, name):
    """内联 GO 富集 (Enrichr, 小鼠库)。gseapy 没装或联网失败 -> 返回 None。"""
    try:
        import gseapy as gp
    except ImportError:
        print(f"  [GO] 未装 gseapy, {name} 跳过 (pip install 'gseapy>=1.0,<1.1')")
        return None
    if len(gene_list) < 10:
        print(f"  [GO] {name} 基因数 {len(gene_list)}<10, 跳过")
        return None
    gene_sets = ["GO_Biological_Process_2021",
                 "MGI_Mammalian_Phenotype_Level_4_2021"]
    all_res = []
    for gs in gene_sets:
        try:
            enr = gp.enrichr(gene_list=list(gene_list), gene_sets=gs,
                             organism="mouse", background=list(background),
                             outdir=None, no_plot=True)
            r = enr.results.copy(); r["library"] = gs
            all_res.append(r)
        except TypeError:
            try:
                enr = gp.enrichr(gene_list=list(gene_list), gene_sets=gs,
                                 outdir=None, no_plot=True)
                r = enr.results.copy(); r["library"] = gs
                all_res.append(r)
            except Exception as e:
                print(f"  [GO] {name}/{gs} 失败: {e}")
        except Exception as e:
            print(f"  [GO] {name}/{gs} 失败 (联网?): {e}")
    if not all_res:
        return None
    res = pd.concat(all_res, ignore_index=True)
    pcol = "Adjusted P-value" if "Adjusted P-value" in res.columns else "P-value"
    res = res.sort_values(pcol).reset_index(drop=True)
    res["_p"] = res[pcol]
    # 顺便存 CSV 供复核
    res.to_csv(output_dir + f"tables/GO_{name}.csv", index=False)
    return res


def add_label(ax, lab):
    ax.text(-0.14, 1.10, lab, transform=ax.transAxes,
            fontsize=20, fontweight="bold", va="top")


# ====================================================================
# 加载 + 计算 (全局, 供所有 panel 函数共用)
# ====================================================================
print("加载数据...")
T = {}
for name in ALL_TISSUES:
    p = TISSUES[name]["path"]
    if not os.path.exists(p):
        continue
    # 只为 skin 加载 dyn (活性曲线只画 skin 的毛囊 TF; dyn 表很大)
    grn = load_grn(p, load_dyn=(name == "skin"))
    if "avg" not in grn:
        continue
    grn["avg"]["TF"] = grn["avg"]["TF"].map(normalize_mouse_symbol)
    grn["avg"]["Gene"] = grn["avg"]["Gene"].map(normalize_mouse_symbol)
    grn["pairs"] = get_pairs(grn["avg"])
    grn["target_sets"] = get_target_sets(grn["avg"])
    grn["n_cells"] = grn["adata"].n_obs if "adata" in grn else np.nan
    grn["n_genes"] = grn["adata"].n_vars if "adata" in grn else np.nan
    if "rg" in grn:
        grn["active_ivs"] = get_active_ivs(grn["rg"])
    T[name] = grn
    print(f"  {name}: {len(grn['pairs']):,} 边, {grn['n_cells']} cells, {grn['n_genes']} genes"
          + (", dyn已加载" if "dyn" in grn else ""))

avail = [t for t in ALL_TISSUES if t in T]

# 层面一
topn_edges = min(len(T[t]["pairs"]) for t in avail)
topn_sets = {t: get_topN_pairs(T[t]["avg"], topn_edges) for t in avail}
pair_jac = pd.DataFrame(0.0, index=avail, columns=avail)
for t1 in avail:
    for t2 in avail:
        s1, s2 = topn_sets[t1], topn_sets[t2]
        pair_jac.loc[t1, t2] = len(s1 & s2) / len(s1 | s2) if (s1 | s2) else 0
s_skin, s_brain = T["skin"]["pairs"], T["brain"]["pairs"]
shared_sb, skin_only, brain_only = s_skin & s_brain, s_skin - s_brain, s_brain - s_skin

# 层面二
tissue_ivs = {t: T[t]["active_ivs"] for t in avail if "active_ivs" in T[t]}
spec_raw = peak_spec_coord(tissue_ivs)
n_t = len(tissue_ivs)
if n_t >= 3:
    peak_spec = {"1 tissue (specific)": spec_raw.get(1, 0), "2 tissues": spec_raw.get(2, 0),
                 f"{n_t} tissues (shared)": sum(v for kk, v in spec_raw.items() if kk >= 3)}
else:
    peak_spec = {"1 tissue (specific)": spec_raw.get(1, 0),
                 "2 tissues (shared)": spec_raw.get(2, 0)}

# 层面三
div_df = identity_divergence(T["skin"]["target_sets"], T["brain"]["target_sets"])
ranked = div_df.reset_index()
ranked["rank"] = range(1, len(ranked) + 1)
total = len(ranked)
known_low = {kt.lower() for kt in SKIN_TFS}
low2rank = {str(t).lower(): r for t, r in zip(ranked["TF"], ranked["rank"])}
spread = np.percentile(ranked["divergence"], 90) - np.percentile(ranked["divergence"], 10)

# 验证 (超几何) — 供正文
K = sum(1 for tf in SKIN_TFS if tf.lower() in {str(x).lower() for x in ranked["TF"]})
top30_low = {str(t).lower() for t in ranked["TF"].head(30)}
hits = [tf for tf in SKIN_TFS if tf.lower() in top30_low]
k = len(hits)
pval = hypergeom.sf(k - 1, total, K, 30) if (K > 0 and k > 0) else 1.0
verdict = "discovery" if pval < 0.05 else "diffuse"

# GO (内联富集, 不依赖外部 CSV)
skin_genes = set(g for _, g in s_skin)
brain_genes = set(g for _, g in s_brain)
go_bg = sorted(skin_genes | brain_genes)
go_skin_genes = sorted(skin_genes - brain_genes)
go_shared_genes = sorted(skin_genes & brain_genes)
print("GO 富集 (内联, 联网 Enrichr)...")
go_skin = run_go_enrichment(go_skin_genes, go_bg, "skin_specific")
go_shared = run_go_enrichment(go_shared_genes, go_bg, "shared")

# 活性曲线 (count 版, 只对出现在 skin 的毛囊 TF; 不用权重)
print("计算毛囊 TF 靶标数活性曲线 (count, 沿伪时序)...")
skin_dyn = T["skin"].get("dyn")
# 统一 dyn 的 TF 命名
if skin_dyn is not None:
    skin_dyn = skin_dyn.copy()
    skin_dyn["TF"] = skin_dyn["TF"].map(normalize_mouse_symbol)
    skin_dyn["Gene"] = skin_dyn["Gene"].map(normalize_mouse_symbol)
activity_curves = compute_count_activity_curves(skin_dyn, SKIN_TFS, n_bins=20)
print(f"  得到 {len(activity_curves)} 个毛囊 TF 的曲线: {list(activity_curves.keys())}")

# Nfib subnetwork 数据
a_set = T["skin"]["target_sets"].get(REP_TF, set())
b_set = T["brain"]["target_sets"].get(REP_TF, set())
shared_t, skin_t, brain_t = a_set & b_set, a_set - b_set, b_set - a_set


# ====================================================================
# 各 panel 绘制函数 (主图与单图共用, 保证一致)
# ====================================================================
def draw_A(fig, subspec, label=True):
    """伪时序 UMAP — 需要 subgridspec, 特殊处理。"""
    gsA = subspec.subgridspec(1, len(avail), wspace=0.12)
    for i, t in enumerate(avail):
        ax = fig.add_subplot(gsA[i])
        if i == 0 and label:
            add_label(ax, "A")
        d = T[t]
        if "adata" in d and hasattr(d["adata"], "obsm") and "X_umap" in d["adata"].obsm:
            a = d["adata"]; um = a.obsm["X_umap"]
            if "pseudotime" in a.obs.columns:
                sc = ax.scatter(um[:, 0], um[:, 1], c=a.obs["pseudotime"].values,
                                cmap="viridis", s=4, alpha=0.7, rasterized=True)
                if i == len(avail) - 1:
                    plt.colorbar(sc, ax=ax, shrink=0.55, pad=0.02, label="pt")
            else:
                ax.scatter(um[:, 0], um[:, 1], s=4, c="gray", alpha=0.5, rasterized=True)
        else:
            ax.text(0.5, 0.5, "No UMAP", transform=ax.transAxes, ha="center")
        ax.set_title(t, fontsize=16, fontweight="bold", color=TISSUES[t]["color"])
        ax.set_xticks([]); ax.set_yticks([])


def draw_B(ax, label=True):
    if label:
        add_label(ax, "B")
    sns.heatmap(pair_jac.loc[avail, avail].astype(float), ax=ax, annot=True, fmt=".3f",
                cmap="YlOrRd", vmin=0, vmax=0.5, square=True,
                annot_kws={"size": 18},
                cbar_kws={"label": "Jaccard (balanced)", "shrink": 0.7})
    ax.set_title(f"Skin vs brain TF-Gene overlap\n(top {topn_edges:,} edges, balanced)")


def draw_C(ax, label=True):
    if label:
        add_label(ax, "C")
    cats = {"shared": len(shared_sb), "skin-spec": len(skin_only), "brain-spec": len(brain_only)}
    bars = ax.bar(range(3), list(cats.values()),
                  color=["#999999", SKIN_C, BRAIN_C], edgecolor="white", alpha=0.85)
    ax.set_xticks(range(3)); ax.set_xticklabels(list(cats.keys()), fontsize=16)
    ymax = max(cats.values())
    for b, v in zip(bars, cats.values()):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + ymax*0.01,
                f"{v:,}", ha="center", fontsize=16)
    ax.set_ylabel("TF-Gene edges")
    ax.set_title("Shared vs tissue-specific edges\n(skin vs brain)")
    ax.spines[["top", "right"]].set_visible(False)


def draw_D(ax, label=True):
    if label:
        add_label(ax, "D")
    if peak_spec and sum(peak_spec.values()) > 0:
        lp, vp = list(peak_spec.keys()), list(peak_spec.values())
        ax.pie(vp, autopct="%1.1f%%", startangle=90, pctdistance=0.75,
               colors=["#E41A1C", "#FF7F00", "#2166AC"][:len(vp)], textprops={"fontsize": 9})
        ax.legend(lp, fontsize=16, loc="lower left", bbox_to_anchor=(-0.12, -0.05))
    ax.set_title("Peak tissue specificity\n(coordinate overlap)")


def draw_G(ax, label=True):
    if label:
        add_label(ax, "G")
    allspec = ranked["divergence"].values
    parts = ax.violinplot([allspec], positions=[0], showmedians=True, widths=0.7)
    for pc in parts["bodies"]:
        pc.set_facecolor("#CFCFCF"); pc.set_alpha(0.6)
    np.random.seed(0)
    # 收集毛囊 TF 的 (点位置, divergence), 按 div 排序后纵向错开标签
    pts = []
    for tf in SKIN_TFS:
        r = low2rank.get(tf.lower())
        if r:
            sv = ranked.loc[ranked["rank"] == r, "divergence"].values
            if len(sv):
                px = np.random.uniform(-0.12, 0.12)
                pts.append((tf, px, sv[0]))
    pts.sort(key=lambda x: x[2])   # 按 divergence 升序
    n = len(pts)
    if n:
        ymin, ymax = allspec.min(), allspec.max()
        # 标签均匀分布在右侧 (纵向错开)
        label_ys = np.linspace(ymin, ymax, n)
        label_x = 0.45
        for (tf, px, py), ly in zip(pts, label_ys):
            ax.scatter(px, py, s=55, color=SKIN_C, edgecolor="black", zorder=5)
            # 引导线: 从点连到右侧标签
            ax.annotate(tf, xy=(px, py), xytext=(label_x, ly),
                        fontsize=16, va="center", ha="left",
                        arrowprops=dict(arrowstyle="-", color="#888", lw=0.6,
                                        connectionstyle="arc3,rad=0.1"))
    ax.set_xlim(-0.5, 0.9)
    ax.set_xticks([0]); ax.set_xticklabels(["all TFs"])
    ax.set_ylabel("identity divergence (1-Jaccard)")
    ax.set_title(f"Distribution of TF rewiring\n(spread 90-10%={spread:.3f}; ★=follicle TF)",
                 fontsize=16)
    ax.spines[["top", "right"]].set_visible(False)


def draw_H(ax, label=True):
    if label:
        add_label(ax, "H")
    np.random.seed(1)
    # 为每个靶基因分配一个固定位置, 并从中心连线 (每类只连部分避免糊)
    MAX_EDGES_PER_GROUP = 40   # 每类最多画的连线数 (点全画, 线只画部分)

    def place_and_draw(genes, color, lab, draw_edges=True):
        n = len(genes)
        if n == 0:
            return
        ang = np.random.uniform(0, 2 * np.pi, n)
        rad = np.random.uniform(0.35, 1.05, n)
        xs, ys = rad * np.cos(ang), rad * np.sin(ang)
        # 连线: 只连前 MAX_EDGES_PER_GROUP 个 (避免糊成一团)
        if draw_edges:
            ne = min(n, MAX_EDGES_PER_GROUP)
            for i in range(ne):
                ax.plot([0, xs[i]], [0, ys[i]], color=color, lw=0.4,
                        alpha=0.25, zorder=1)
        # 点: 全画
        ax.scatter(xs, ys, s=16, c=color, alpha=0.7, edgecolor="none",
                   zorder=3, label=f"{lab} ({n})")

    place_and_draw(shared_t, "#999999", "shared")
    place_and_draw(skin_t, SKIN_C, "skin-specific")
    place_and_draw(brain_t, BRAIN_C, "brain-specific")
    ax.scatter([0], [0], s=260, c="black", marker="*", zorder=10)
    ax.annotate(REP_TF, (0, 0), fontsize=16, fontweight="bold", ha="center", va="bottom",
                xytext=(0, 10), textcoords="offset points", zorder=11)
    ax.set_xlim(-1.3, 1.3); ax.set_ylim(-1.3, 1.3)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_aspect("equal")
    ax.legend(fontsize=16, loc="upper right")
    ax.set_title(f"Target rewiring of follicle TF {REP_TF}\n"
                 f"(skin vs brain; edges shown for ≤{MAX_EDGES_PER_GROUP}/group)",
                 fontsize=16)


def draw_J(ax, label=True):
    if label:
        add_label(ax, "J")
    if go_skin is not None:
        sk = go_skin.head(8).copy()
        sk["nlp"] = -np.log10(sk["_p"]); sk = sk.iloc[::-1]
        yy = np.arange(len(sk))
        ax.barh(yy, sk["nlp"].values, color=SKIN_C, alpha=0.85)
        if go_shared is not None:
            best_sh = go_shared["_p"].min()
            ax.text(0.98, 0.03, f"shared targets:\nbest adj.P={best_sh:.2f} (n.s.)",
                    transform=ax.transAxes, ha="right", va="bottom", fontsize=16,
                    color="#666", bbox=dict(boxstyle="round", fc="#EEE", ec="#999"))
        ax.set_yticks(yy); ax.set_yticklabels([t[:30] for t in sk["Term"]], fontsize=14)
        ax.axvline(-np.log10(0.05), color="gray", ls="--", lw=0.8)
        ax.set_xlabel("-log10 adjusted P", fontsize=16)
        ax.set_title("GO: skin-specific targets enrich skin biology\n"
                     "(shared targets: no tissue term)", fontsize=16)
        ax.spines[["top", "right"]].set_visible(False)
    else:
        ax.text(0.5, 0.5, "GO CSV not found", transform=ax.transAxes, ha="center")
        ax.axis("off")


def draw_K(ax, label=True):
    if label:
        add_label(ax, "K")
    only_s = len(skin_genes - brain_genes)
    only_b = len(brain_genes - skin_genes)
    both = len(skin_genes & brain_genes)
    ax.add_patch(Circle((-0.35, 0), 0.6, alpha=0.4, color=SKIN_C))
    ax.add_patch(Circle((0.35, 0), 0.6, alpha=0.4, color=BRAIN_C))
    ax.text(-0.62, 0, f"skin\n{only_s}", ha="center", va="center", fontsize=16, fontweight="bold")
    ax.text(0.62, 0, f"brain\n{only_b}", ha="center", va="center", fontsize=16, fontweight="bold")
    ax.text(0, 0, f"{both}", ha="center", va="center", fontsize=16, fontweight="bold")
    ax.set_xlim(-1.3, 1.3); ax.set_ylim(-0.9, 0.9)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_aspect("equal")
    ax.set_title("Target gene overlap\n(skin vs brain)", fontsize=16)
    for sp_ in ax.spines.values():
        sp_.set_visible(False)


def draw_L(fig, subspec, label=True):
    """毛囊 TF 靶标数(count)活性曲线 — 拆成小多图, 每个 TF 一格。不用权重。"""
    if not activity_curves:
        ax = fig.add_subplot(subspec)
        if label:
            add_label(ax, "L")
        ax.text(0.5, 0.5, "No dynamic (dyn) data", transform=ax.transAxes,
                ha="center", va="center", color="#888")
        ax.set_title("TF target-count activity (skin)")
        ax.axis("off")
        return
    tfs = list(activity_curves.keys())
    n = len(tfs)
    ncol = 4
    nrow = int(np.ceil(n / ncol))
    n_bins = len(next(iter(activity_curves.values())))
    x = np.linspace(0, 1, n_bins)
    # 平滑用的密集 x 轴
    x_smooth = np.linspace(0, 1, 200)
    try:
        from scipy.interpolate import make_interp_spline
        _HAS_SCIPY_SPLINE = True
    except ImportError:
        _HAS_SCIPY_SPLINE = False

    def smooth_curve(y):
        """样条平滑; scipy 不可用则退回轻度滑动平均。"""
        if _HAS_SCIPY_SPLINE and len(y) >= 4:
            try:
                spl = make_interp_spline(x, y, k=3)
                ys = spl(x_smooth)
                return x_smooth, np.clip(ys, 0, 1.05)  # 限幅避免样条过冲
            except Exception:
                pass
        # 退回: 滑动平均(窗口3)
        k = np.ones(3) / 3
        ys = np.convolve(y, k, mode="same")
        return x, ys

    inner = subspec.subgridspec(nrow, ncol, wspace=0.35, hspace=0.55)
    cmap = plt.cm.tab10(np.linspace(0, 1, n))
    for i, (tf, c) in enumerate(zip(tfs, cmap)):
        ax = fig.add_subplot(inner[i // ncol, i % ncol])
        if i == 0 and label:
            add_label(ax, "L")
        curve = activity_curves[tf]
        xs, ys = smooth_curve(curve)
        ax.plot(xs, ys, lw=1.8, color=c)
        ax.fill_between(xs, 0, ys, color=c, alpha=0.15)
        ax.set_title(tf, fontsize=16, fontweight="bold")
        ax.set_ylim(-0.05, 1.15)
        ax.set_xticks([0, 0.5, 1.0]); ax.tick_params(labelsize=10)
        if i % ncol == 0:
            ax.set_ylabel("target\ncount (norm)", fontsize=12)
        if i // ncol == nrow - 1:
            ax.set_xlabel("pseudotime", fontsize=12)
        ax.spines[["top", "right"]].set_visible(False)


# ====================================================================
# 绘制主图 (4x2)
# ====================================================================
print("绘制主图...")
fig = plt.figure(figsize=(20, 30))
fig.patch.set_facecolor("white")
gs = gridspec.GridSpec(5, 2, figure=fig, hspace=0.45, wspace=0.28,
                       left=0.08, right=0.95, top=0.97, bottom=0.03)

draw_A(fig, gs[0, 0])
draw_B(fig.add_subplot(gs[0, 1]))
draw_C(fig.add_subplot(gs[1, 0]))
draw_D(fig.add_subplot(gs[1, 1]))
draw_G(fig.add_subplot(gs[2, 0]))
draw_H(fig.add_subplot(gs[2, 1]))
draw_J(fig.add_subplot(gs[3, 0]))
draw_K(fig.add_subplot(gs[3, 1]))
draw_L(fig, gs[4, :])  # L 跨整行 (内部 8 个小图)

fig.savefig(output_dir + "Case3_main_figure.png", dpi=300, bbox_inches="tight")
fig.savefig(output_dir + "Case3_main_figure.pdf", bbox_inches="tight")
plt.close(fig)
print(f"  主图: {output_dir}Case3_main_figure.pdf")


# ====================================================================
# 绘制各子图 (单独保存, 调用同一函数)
# ====================================================================
print("绘制各子图...")

# A 特殊 (多 UMAP), 单独处理
figA = plt.figure(figsize=(6 * len(avail), 5.5))
gsa = gridspec.GridSpec(1, 1, figure=figA)
draw_A(figA, gsa[0, 0], label=False)
figA.savefig(output_dir + "panels/panel_A.png", dpi=300, bbox_inches="tight")
figA.savefig(output_dir + "panels/panel_A.pdf", bbox_inches="tight")
plt.close(figA)

# B-K 通用单图
single_panels = {
    "B": (draw_B, (9, 7.5)),
    "C": (draw_C, (9, 7.5)),
    "D": (draw_D, (9, 7.5)),
    "G": (draw_G, (8, 9)),
    "H": (draw_H, (9, 9)),
    "J": (draw_J, (11, 7.5)),
    "K": (draw_K, (9, 6)),
}
for letter, (fn, figsize) in single_panels.items():
    f = plt.figure(figsize=figsize)
    ax = f.add_subplot(111)
    fn(ax, label=False)
    f.savefig(output_dir + f"panels/panel_{letter}.png", dpi=300, bbox_inches="tight")
    f.savefig(output_dir + f"panels/panel_{letter}.pdf", bbox_inches="tight")
    plt.close(f)
    print(f"  panel_{letter}")

# L 单图 (小多图, 签名特殊)
figL = plt.figure(figsize=(16, 8))
gsl = gridspec.GridSpec(1, 1, figure=figL)
draw_L(figL, gsl[0, 0], label=False)
figL.suptitle("Follicle-TF target-count activity along pseudotime (skin)\n"
              "count-based; network weights saturated and not used", fontsize=16)
figL.savefig(output_dir + "panels/panel_L.png", dpi=300, bbox_inches="tight")
figL.savefig(output_dir + "panels/panel_L.pdf", bbox_inches="tight")
plt.close(figL)
print("  panel_L")


# ====================================================================
# 导出表
# ====================================================================
pair_jac.to_csv(output_dir + "tables/jaccard_skin_brain.csv")
div_df.to_csv(output_dir + "tables/skin_brain_identity_divergence.csv")
rank_rows = [{"TF": tf, "rank": low2rank.get(tf.lower()),
              "in_top30": tf.lower() in top30_low} for tf in SKIN_TFS]
pd.DataFrame(rank_rows).to_csv(output_dir + "tables/known_follicle_TF_ranks.csv", index=False)
pd.DataFrame([{"category": kk, "n_peaks": vv} for kk, vv in peak_spec.items()]).to_csv(
    output_dir + "tables/peak_specificity.csv", index=False)

# 活性曲线数据导出
if activity_curves:
    pd.DataFrame(activity_curves).to_csv(
        output_dir + "tables/follicle_TF_activity_curves.csv", index=False)


# ====================================================================
# 分析打印 (写正文用)
# ====================================================================
only_s = len(skin_genes - brain_genes)
only_b = len(brain_genes - skin_genes)
both = len(skin_genes & brain_genes)
peak_total = sum(peak_spec.values())

print("\n" + "=" * 64)
print("分析结果 (写正文用)")
print("=" * 64)
print(f"[层面一] skin-brain Jaccard = {pair_jac.loc['skin','brain']:.3f} "
      f"(归一化 top {topn_edges:,} 边)")
print(f"         边分解: shared={len(shared_sb):,}, "
      f"skin-spec={len(skin_only):,}, brain-spec={len(brain_only):,}")
print(f"[层面二] peak 特异性 (skin/brain 两器官口径):")
for kk, vv in peak_spec.items():
    print(f"         {kk}: {vv:,} ({vv/peak_total*100:.1f}%)" if peak_total else f"         {kk}: {vv}")
print(f"[层面三] 分化度: N={total} TFs, 中位={div_df['divergence'].median():.3f}, "
      f"spread(90-10%)={spread:.3f}")
print(f"         毛囊TF: K={K}, top30命中 k={k}: {hits}, p={pval:.2e}, verdict={verdict}")
print(f"         毛囊TF排名:")
for tf in SKIN_TFS:
    r = low2rank.get(tf.lower())
    print(f"           {tf:10s}: {r}/{total}" if r else f"           {tf:10s}: 不在共有TF池")
print(f"         Nfib subnetwork: shared={len(shared_t)}, "
      f"skin-spec={len(skin_t)}, brain-spec={len(brain_t)}")
print(f"         靶基因Euler: skin-only={only_s}, brain-only={only_b}, shared={both}")
if go_skin is not None:
    print(f"[GO] skin特异 top: {go_skin.iloc[0]['Term'][:45]} p={go_skin.iloc[0]['_p']:.2e}")
    if go_shared is not None:
        print(f"     shared best p={go_shared['_p'].min():.3f} (阴性对照)")
else:
    print("[GO] 未找到 CSV, 先跑 case3_go_enrichment.py")
print("\n完成。主图 + 各子图(panels/) + 表(tables/) 已保存。")

加载数据...
  skin: 526,742 边, 1000 cells, 1999 genes, dyn已加载
  brain: 674,686 边, 1000 cells, 2000 genes
GO 富集 (内联, 联网 Enrichr)...
  [GO] skin_specific/GO_Biological_Process_2021 失败 (联网?): HTTPSConnectionPool(host='maayanlab.cloud', port=443): Max retries exceeded with url: /Enrichr/datasetStatistics (Caused by ResponseError('too many 504 error responses'))
计算毛囊 TF 靶标数活性曲线 (count, 沿伪时序)...
  得到 8 个毛囊 TF 的曲线: ['Foxq1', 'Gata3', 'Hoxc13', 'Lef1', 'Nfib', 'Sox9', 'Tcf7', 'Vdr']
绘制主图...
  主图: /home/wuyan/dygmamba_project/DRIMA/data/case3/Case3_main_figure.pdf
绘制各子图...
  panel_B
  panel_C
  panel_D
  panel_G
  panel_H
  panel_J
  panel_K
  panel_L

分析结果 (写正文用)
[层面一] skin-brain Jaccard = 0.096 (归一化 top 526,742 边)
         边分解: shared=111,301, skin-spec=415,441, brain-spec=563,385
[层面二] peak 特异性 (skin/brain 两器官口径):
         1 tissue (specific): 9,664 (92.5%)
         2 tissues (shared): 786 (7.5%)
[层面三] 分化度: N=837 TFs, 中位=0.928, spread(90-10%)=0.105
         毛囊TF: K=8, top30命中 k=0: [], p=1.00e+00